# GPU-Fuzzy Trading Pipeline in VS Code Colab

This notebook is meant to run on a Colab server connected from the VS Code Colab extension.

Before running the cells:
- Connect the notebook to a Colab server.
- Either upload the entire `trading_platform/` folder to that server using Explorer > Upload to Colab, or let the bootstrap cell clone the private GitHub repo.
- If the folder lands somewhere else, set `PROJECT_ROOT` in the bootstrap cell to the uploaded path.
- If `train.csv` and `test.csv` live in Google Drive, the bootstrap cell will use `/content/drive/MyDrive/train.csv` and `/content/drive/MyDrive/test.csv` and link them into the repo `data/` folder.

The bootstrap cell mounts Google Drive when needed.

If a previous Colab run patched `gpu_fuzzy_trader/evolution/numba_ops.py` on the server, re-clone the repo or delete `/content/trading_platform` and restart the kernel before rerunning.


In [6]:
import os
from getpass import getpass

github_token = os.environ.get("GITHUB_TOKEN", "").strip()
if not github_token:
    github_token = getpass("GitHub classic PAT (optional, press Enter to skip): " ).strip()
if github_token:
    os.environ["GITHUB_TOKEN"] = github_token


In [7]:
from base64 import b64encode
from pathlib import Path
from urllib.parse import urlsplit
import os
import shutil
import subprocess
import sys


PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "").strip() or None
DEFAULT_GITHUB_REPO_URL = "https://github.com/m-danaee/trading_platform.git"
GITHUB_REPO_URL = os.environ.get("GITHUB_REPO_URL", DEFAULT_GITHUB_REPO_URL).strip() or DEFAULT_GITHUB_REPO_URL
GITHUB_BRANCH = os.environ.get("GITHUB_BRANCH", "main").strip() or "main"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip() or None
GITHUB_CLONE_DIR = Path(os.environ.get("GITHUB_CLONE_DIR", "/content/trading_platform")).expanduser()
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
os.environ["TRAIN_CSV_PATH"] = "/content/drive/MyDrive/train.csv"
os.environ["TEST_CSV_PATH"] = "/content/drive/MyDrive/test.csv"
DATA_ROOT = os.environ.get("DATA_ROOT", "").strip() or None
TRAIN_CSV_PATH = os.environ.get("TRAIN_CSV_PATH", "").strip() or None
TEST_CSV_PATH = os.environ.get("TEST_CSV_PATH", "").strip() or None


def _is_project_root(path: Path) -> bool:
    return (path / "gpu_fuzzy_trader").is_dir() and (path / "requirements.txt").is_file()


def _discover_project_root() -> Path | None:
    if PROJECT_ROOT is not None:
        candidate = Path(PROJECT_ROOT).expanduser()
        if _is_project_root(candidate):
            return candidate.resolve()

    for candidate in [Path.cwd(), Path("/content/trading_platform"), Path("/content")]:
        if _is_project_root(candidate):
            return candidate.resolve()

    content_root = Path("/content")
    if content_root.exists():
        for requirements_file in content_root.rglob("requirements.txt"):
            candidate = requirements_file.parent
            if _is_project_root(candidate):
                return candidate.resolve()

    return None


def _repo_url_has_credentials(repo_url: str) -> bool:
    parsed = urlsplit(repo_url)
    return bool(parsed.username or parsed.password)


def _ensure_drive_mounted() -> Path | None:
    drive_root = Path("/content/drive")
    if drive_root.exists():
        return drive_root

    try:
        from google.colab import drive
    except Exception:
        return None

    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        return None

    return drive_root if drive_root.exists() else None


def _path_has_csv_pair(root: Path) -> bool:
    return (root / "train.csv").is_file() and (root / "test.csv").is_file()


def _discover_dataset_paths(project_root: Path) -> tuple[Path, Path] | None:
    explicit_train = Path(TRAIN_CSV_PATH).expanduser() if TRAIN_CSV_PATH else None
    explicit_test = Path(TEST_CSV_PATH).expanduser() if TEST_CSV_PATH else None
    if explicit_train and explicit_test and explicit_train.is_file() and explicit_test.is_file():
        return explicit_train.resolve(), explicit_test.resolve()

    if DATA_ROOT:
        candidate = Path(DATA_ROOT).expanduser()
        if _path_has_csv_pair(candidate):
            return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

    for candidate in [
        project_root / "data",
        project_root,
        Path("/content/trading_platform/data"),
        Path("/content/trading_platform"),
    ]:
        if _path_has_csv_pair(candidate):
            return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

    drive_root = _ensure_drive_mounted()
    if drive_root is not None:
        drive_candidates = [
            drive_root / "MyDrive",
            drive_root / "My Drive",
            drive_root / "Shareddrives",
            drive_root / "Shared drives",
            drive_root,
        ]
        for candidate in drive_candidates:
            if _path_has_csv_pair(candidate):
                return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

        for candidate in drive_candidates:
            if not candidate.exists():
                continue
            for train_file in candidate.rglob("train.csv"):
                candidate_dir = train_file.parent
                if (candidate_dir / "test.csv").is_file():
                    return train_file.resolve(), (candidate_dir / "test.csv").resolve()

    content_root = Path("/content")
    if content_root.exists():
        for train_file in content_root.rglob("train.csv"):
            candidate = train_file.parent
            if (candidate / "test.csv").is_file():
                return train_file.resolve(), (candidate / "test.csv").resolve()

    return None


def _ensure_repo_data_files(project_root: Path, train_csv_path: Path, test_csv_path: Path) -> None:
    data_dir = project_root / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    targets = {
        data_dir / "train.csv": train_csv_path,
        data_dir / "test.csv": test_csv_path,
    }

    for target, source in targets.items():
        if target.exists():
            continue
        try:
            target.symlink_to(source)
        except Exception:
            shutil.copy2(source, target)


def _repair_stale_numba_ops(project_root: Path) -> str:
    """Remove broken JIT empty branches left by older Colab notebook patches."""
    numba_ops_path = project_root / "gpu_fuzzy_trader" / "evolution" / "numba_ops.py"
    if not numba_ops_path.is_file():
        return "numba_ops.py not found"

    text = numba_ops_path.read_text(encoding="utf-8")
    original = text
    for block in (
        "        if n == 0:\n            return [[]]\n\n",
        "        if n == 0:\n            return [[-1]]\n\n",
        "        if n == 0:\n            return [[]]\n",
        "        if n == 0:\n            return [[-1]]\n",
    ):
        text = text.replace(block, "")

    wrapper_guard = (
        "    if obj.shape[0] == 0:\n"
        "        return [[]]\n"
    )
    if wrapper_guard not in text and "def non_dominated_sort" in text:
        needle = "    obj = np.asarray(objectives, dtype=np.float64)\n"
        if needle in text:
            text = text.replace(
                needle,
                needle + wrapper_guard,
                1,
            )

    if text != original:
        numba_ops_path.write_text(text, encoding="utf-8")
        return "repaired stale JIT empty branch in numba_ops.py"

    jit_parts = text.split("def _non_dominated_sort_numba", 1)
    if len(jit_parts) > 1:
        jit_body = jit_parts[1].split("\n    @", 1)[0].split("\n    def ", 1)[0]
        if "if n == 0:" in jit_body:
            raise RuntimeError(
                "numba_ops.py still has a Numba-incompatible empty branch. "
                "Delete /content/trading_platform and rerun bootstrap, or push the latest repo and git pull."
            )

    return "numba_ops.py ok"


def _try_git_pull(project_root: Path) -> str:
    git_dir = project_root / ".git"
    if not git_dir.is_dir():
        return "not a git repo"
    result = subprocess.run(
        ["git", "-C", str(project_root), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return (result.stdout or "").strip() or "git pull ok"
    return f"git pull skipped: {(result.stderr or result.stdout or '').strip()}"


def _looks_like_auth_failure(output: str) -> bool:
    text = output.lower()
    return any(marker in text for marker in [
        'authentication failed',
        'could not read username',
        'repository not found',
        'terminal prompts disabled',
        'support for password authentication was removed',
        'access denied',
    ])


def _basic_auth_header(token: str) -> str:
    token_bytes = f'x-access-token:{token}'.encode('utf-8')
    encoded = b64encode(token_bytes).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'


def _run_git_clone(repo_url: str, target_dir: Path, token: str | None = None) -> subprocess.CompletedProcess[str]:
    clone_cmd = [
        'git',
        'clone',
        '--depth',
        '1',
        '--branch',
        GITHUB_BRANCH,
        repo_url,
        str(target_dir),
    ]
    clone_env = os.environ.copy()
    clone_env['GIT_TERMINAL_PROMPT'] = '0'
    clone_kwargs = {'capture_output': True, 'text': True, 'env': clone_env}
    if token:
        clone_cmd = [
            'git',
            '-c',
            f'http.extraheader={_basic_auth_header(token)}',
            'clone',
            '--depth',
            '1',
            '--branch',
            GITHUB_BRANCH,
            repo_url,
            str(target_dir),
        ]
    return subprocess.run(clone_cmd, **clone_kwargs)


def _clone_private_repo(repo_url: str, target_dir: Path) -> Path:

    if not repo_url:
        raise ValueError("GITHUB_REPO_URL is empty.")

    if target_dir.exists():
        if _is_project_root(target_dir):
            _try_git_pull(target_dir)
            return target_dir.resolve()
        if target_dir.is_dir() and any(target_dir.iterdir()):
            raise FileExistsError(
                f"{target_dir} already exists and is not the project root. "
                "Change GITHUB_CLONE_DIR or remove the conflicting directory."
            )
    target_dir.mkdir(parents=True, exist_ok=True)

    clone_result = _run_git_clone(repo_url, target_dir, GITHUB_TOKEN)
    if clone_result.returncode != 0:
        combined_output = (clone_result.stdout or '') + '\n' + (clone_result.stderr or '')
        hint = ""
        if _looks_like_auth_failure(combined_output):
            hint = (
                'This looks like a GitHub auth problem. For a classic PAT, use a token '
                'with repo read access, store it in Cell 1, and make sure the repository URL is correct.'
            )
        raise RuntimeError(
            'Git clone failed.\n'
            f'{combined_output.strip() or "(no git output)"}\n'
            f'{hint}'
        )

    return target_dir.resolve()


project_root = _discover_project_root()
if project_root is None and GITHUB_REPO_URL:
    project_root = _clone_private_repo(GITHUB_REPO_URL, GITHUB_CLONE_DIR)

if project_root is None:
    content_entries = []
    content_root = Path("/content")
    if content_root.exists():
        for path in sorted(content_root.iterdir()):
            content_entries.append(f"{path.name}/" if path.is_dir() else path.name)
    raise FileNotFoundError(
        "Could not find the project folder on the Colab server.\n"
        "Either upload the entire trading_platform/ folder, or set GITHUB_REPO_URL "
        "(and store your classic PAT in Cell 1 if the repo is private) and rerun it.\n"
        f"/content currently contains: {', '.join(content_entries) if content_entries else '(empty or unavailable)'}"
    )

resolved_dataset_paths = _discover_dataset_paths(project_root)
if resolved_dataset_paths is None:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv.\n"
        "Either set DATA_ROOT or TRAIN_CSV_PATH / TEST_CSV_PATH to the Google Drive location, "
        "or upload both files to the project data/ directory and rerun Cell 2."
    )

train_csv_path, test_csv_path = resolved_dataset_paths
_ensure_repo_data_files(project_root, train_csv_path, test_csv_path)
numba_ops_status = _repair_stale_numba_ops(project_root)
if (project_root / ".git").is_dir():
    git_pull_status = _try_git_pull(project_root)
    numba_ops_status = _repair_stale_numba_ops(project_root)
else:
    git_pull_status = "not a git repo"
os.environ["TRAIN_CSV_PATH"] = str(train_csv_path)
os.environ["TEST_CSV_PATH"] = str(test_csv_path)
if train_csv_path.parent == test_csv_path.parent:
    os.environ["DATA_ROOT"] = str(train_csv_path.parent)

PROJECT_ROOT = project_root
os.environ["PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

existing_pythonpath = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = (
    str(project_root)
    if not existing_pythonpath
    else f"{project_root}{os.pathsep}{existing_pythonpath}"
)

print(f"Project root: {project_root}")
print(f"Python: {sys.executable}")
print(f"GitHub repo URL configured: {bool(GITHUB_REPO_URL)}")
print(f"GitHub URL has embedded credentials: {_repo_url_has_credentials(GITHUB_REPO_URL)}")
print(f"Train CSV: {train_csv_path}")
print(f"Test CSV: {test_csv_path}")
print(f"Numba ops: {numba_ops_status}")
print(f"Git sync: {git_pull_status}")
print(f"Repo data links ready: {(project_root / 'data' / 'train.csv').exists() and (project_root / 'data' / 'test.csv').exists()}")
print(f"Upload detected under /content: {str(project_root).startswith('/content')}")


Project root: /content/trading_platform
Python: /usr/bin/python3
GitHub repo URL configured: True
GitHub URL has embedded credentials: False
Train CSV: /content/drive/MyDrive/train.csv
Test CSV: /content/drive/MyDrive/test.csv
Repo data links ready: True
Upload detected under /content: True


In [8]:
import subprocess

print("Installing dependencies from requirements.txt ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])


Installing dependencies from requirements.txt ...


0

In [9]:
from gpu_fuzzy_trader.run_pipeline import Pipeline_Orchestrator
from gpu_fuzzy_trader import config as cfg

print("Import OK")
print(f"TRAIN_CSV_PATH: {Path(cfg.TRAIN_CSV_PATH).resolve()}")
print(f"TEST_CSV_PATH: {Path(cfg.TEST_CSV_PATH).resolve()}")

Import OK
TRAIN_CSV_PATH: /content/drive/MyDrive/train.csv
TEST_CSV_PATH: /content/drive/MyDrive/test.csv


In [10]:
import subprocess
import sys

RUN_PHASE = None
RESUME = False
OUTPUT_DIR = "/content/drive/MyDrive/trading_platform/outputs"

pipeline_cmd = [
    sys.executable,
    "-u",
    "-m",
    "gpu_fuzzy_trader.run_pipeline",
    "--output",
    OUTPUT_DIR,
]
if RUN_PHASE is not None:
    pipeline_cmd.extend(["--phase", str(RUN_PHASE)])
if RESUME:
    pipeline_cmd.append("--resume")

print("Running:", " ".join(pipeline_cmd), flush=True)

process = subprocess.Popen(
    pipeline_cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

assert process.stdout is not None
for line in iter(process.stdout.readline, ""):
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, pipeline_cmd)

print("\nPipeline finished successfully.", flush=True)


Running: /usr/bin/python3 -u -m gpu_fuzzy_trader.run_pipeline --output /content/drive/MyDrive/trading_platform/outputs
2026-05-26 11:12:52 [INFO] __main__: ============================================================
2026-05-26 11:12:52 [INFO] __main__: GPU-Fuzzy Trading Pipeline — starting
2026-05-26 11:12:52 [INFO] __main__: ============================================================
2026-05-26 11:12:52 [INFO] __main__: Pipeline config: PHASE1 top_k=22 | PHASE2 algo=NSGA3 pop=200 gen=200 | PHASE3 refine pop=100 gen=80 parallel_batch=True gpu=False | PHASE4 trials=1000 wf_splits=2 sampler=nsga2 n_jobs=1
2026-05-26 11:12:52 [INFO] __main__: Output directories ready: /content/drive/MyDrive/trading_platform/outputs, /content/drive/MyDrive/trading_platform/outputs/reports
2026-05-26 11:12:53 [INFO] __main__: Using cached train/validation split from data/train_75.parquet and data/validation_25.parquet
2026-05-26 11:12:53 [INFO] __main__: Running Phase 1: Feature Selection …
2026-05-26 11:

After the pipeline finishes, the artifacts will be saved to `MyDrive/trading_platform/outputs/` on Google Drive. If you uploaded the project with VS Code, open `evaluator_v3.ipynb` from the same folder and point it at the generated `outputs/long.json` and `outputs/short.json` files.
